In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install necessary utilities
!apt-get install wget

# Download the dataset
!wget http://www-i6.informatik.rwth-aachen.de/imageclef/resources/iaprtc12.tgz -O iaprtc12.tgz

# Create a directory for extraction
!mkdir -p /content/drive/MyDrive/dataset

# Extract the downloaded tar file
!tar -xvzf iaprtc12.tgz -C /content/drive/MyDrive/dataset/


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
iaprtc12/images/32/32281.jpg
iaprtc12/images/32/32282.jpg
iaprtc12/images/32/32283.jpg
iaprtc12/images/32/32284.jpg
iaprtc12/images/32/32285.jpg
iaprtc12/images/32/32286.jpg
iaprtc12/images/32/32287.jpg
iaprtc12/images/32/32288.jpg
iaprtc12/images/32/32289.jpg
iaprtc12/images/32/32290.jpg
iaprtc12/images/32/32291.jpg
iaprtc12/images/32/32292.jpg
iaprtc12/images/32/32293.jpg
iaprtc12/images/32/32294.jpg
iaprtc12/images/32/32295.jpg
iaprtc12/images/32/32296.jpg
iaprtc12/images/32/32297.jpg
iaprtc12/images/32/32298.jpg
iaprtc12/images/32/32299.jpg
iaprtc12/images/32/32300.jpg
iaprtc12/images/32/32301.jpg
iaprtc12/images/32/32302.jpg
iaprtc12/images/32/32303.jpg
iaprtc12/images/32/32304.jpg
iaprtc12/images/32/32305.jpg
iaprtc12/images/32/32306.jpg
iaprtc12/images/32/32307.jpg
iaprtc12/images/32/32308.jpg
iaprtc12/images/32/32309.jpg
iaprtc12/images/32/32310.jpg
iaprtc12/images/32/32311.jpg
iaprtc12/images/32/3231

**Les bibliothèques**

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os
from chardet.universaldetector import UniversalDetector
import json
import re
import numpy as np
import pandas as pd
import os
import xml.etree.ElementTree as ET

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


**Extraction des données**

In [ ]:
# Définir le chemin du dataset
dataset_path = '/content/drive/MyDrive/dataset/iaprtc12/annotations_complete_eng'

# Initialiser les listes pour stocker les métadonnées
combined_texts = []  # Pour stocker le texte combiné (titre + description)
file_names = []  # Pour stocker les noms des fichiers
skipped_files = []  # Pour garder trace des fichiers problématiques
supported_extensions = {'.eng'}  # Extension de fichier supportée

# Lire et traiter les fichiers
for dirpath, dirnames, filenames in os.walk(dataset_path):
    for file in filenames:
        file_path = os.path.join(dirpath, file)
        file_extension = os.path.splitext(file)[-1].lower()

        # Vérifier si l'extension du fichier est supportée
        if file_extension in supported_extensions:
            try:
                # Lire le contenu du fichier pour vérifier s'il est bien formé XML
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    raw_data = f.read()

                # Parser le fichier XML
                tree = ET.ElementTree(ET.fromstring(raw_data))
                xml_root = tree.getroot()

                # Extraire le titre
                title_element = xml_root.find('TITLE')
                title = title_element.text.strip() if title_element is not None else "No title available"

                # Extraire la description
                description_element = xml_root.find('DESCRIPTION')
                description = description_element.text.strip() if description_element is not None else "No description available"

                # Combiner le titre et la description
                combined_text = f"{title}. {description}"
                combined_texts.append(combined_text)

                # Ajouter le nom du fichier
                file_names.append(file)

            except ET.ParseError:
                print(f"Skipping invalid XML file: {file_path}")
                skipped_files.append(file_path)
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")
                skipped_files.append(file_path)


In [ ]:
# Créer un DataFrame avec les données
df = pd.DataFrame({
    'File_Name': file_names,
    'Combined_Text': combined_texts
})
df

,File_Name,Combined_Text
0,781.eng,Godchild Diana Alacron Flores. portrait of a d...
1,738.eng,Kids in the Classroom. Excited kids are standi...
2,674.eng,Children Projects in Arequipa. Three social wo...
3,754.eng,Food for El Alto. Tourists are standing on a w...
4,25.eng,The Plaza de Armas. a yellow building with whi...
...,...,...
16810,21684.eng,The Reforestation Project near Quito. many sma...
16811,21959.eng,"Bernhard, Frauke and Andrea in Paracas. a man ..."
16812,21704.eng,Eduardo and Robinson Brito. a boy is lying on ...
16813,21703.eng,Naim Len and Fanny Bora. a woman with a dark g...


**Générer les embeddings**

In [ ]:
# Load embedding model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

In [ ]:
# Générer les embeddings
embeddings = model.encode(df['Combined_Text'].tolist(), show_progress_bar=True)

# Add the embeddings as a new column in the DataFrame
df['Embeddings'] = embeddings.tolist()
print("Embeddings pour les queries générés !")

Batches:   0%|          | 0/526 [00:00<?, ?it/s]

Embeddings pour les queries générés !


**Sauvegarder les embeddings**

In [ ]:
# Save the DataFrame to a CSV file
output_file_path = '/content/drive/My Drive/CLEF_with_embeddings.csv'
df.to_csv(output_file_path, index=False)
print(f"File saved with embeddings at: {output_file_path}")

File saved with embeddings at: /content/drive/My Drive/CLEF_with_embeddings.csv


In [ ]:
# Load the CSV file with embeddings
file_path = '/content/drive/My Drive/CLEF_with_embeddings.csv'
data = pd.read_csv(file_path)
data.head()

,File_Name,Combined_Text,Embeddings
0,781.eng,Godchild Diana Alacron Flores. portrait of a d...,"[-0.11171477288007736, 0.3831174671649933, 0.1..."
1,738.eng,Kids in the Classroom. Excited kids are standi...,"[0.04371282458305359, 0.3194780647754669, -0.1..."
2,674.eng,Children Projects in Arequipa. Three social wo...,"[0.152804434299469, 0.23115581274032593, -0.03..."
3,754.eng,Food for El Alto. Tourists are standing on a w...,"[0.5058570504188538, 0.24792098999023438, 0.13..."
4,25.eng,The Plaza de Armas. a yellow building with whi...,"[0.6194474697113037, 0.4076594412326813, -0.21..."


**Recherche avec Similarité Cosinus et selection des documents pertinent**

In [ ]:
def search_documents(model, df, query, top_n=30, threshold=0.5):
    # Générer l'embedding pour la requête
    query_embedding = model.encode([query])

    # Vérifier si les embeddings sont déjà sous forme de liste numpy dans df
    if isinstance(df['Embeddings'].iloc[0], list):
        embeddings = df['Embeddings'].apply(np.array)
    else:
        # Convertir les embeddings en numpy array s'ils sont sous forme de chaîne JSON
        embeddings = df['Embeddings'].apply(
            lambda x: np.array(json.loads(x)) if isinstance(x, str) else np.array(x)
        )

    # Calcul de la similarité cosinus entre l'embedding de la requête et les embeddings des documents
    df['Cosine_Similarity'] = embeddings.apply(
        lambda x: cosine_similarity([query_embedding[0]], [x])[0][0]
    )

    # Trier les documents par similarité décroissante
    sorted_df = df.sort_values(by='Cosine_Similarity', ascending=False)

    # Sélectionner les n meilleurs documents
    top_documents = sorted_df.head(top_n)[['Combined_Text', 'Cosine_Similarity']]

    # Sélectionner les documents pertinents au-dessus du seuil de similarité
    relevant_documents = sorted_df[sorted_df['Cosine_Similarity'] > threshold][['Combined_Text', 'Cosine_Similarity']]

    top_documents_list = top_documents.values.tolist()
    relevant_documents_list = relevant_documents.values.tolist()

    return top_documents_list, relevant_documents_list

**Calcul des Métriques**

*   **Precision@30**

In [ ]:
def calculate_precision_at_k(retrieved_docs, relevant_docs, k=30):
    # Prendre les k premiers documents récupérés
    top_k_retrieved = retrieved_docs[:k]

    # Calculer le nombre de vrais positifs
    true_positives = len([doc for doc in top_k_retrieved if doc in relevant_docs])

    # Calculer la précision à k
    precision = true_positives / k if k != 0 else 0
    return precision

*   **Recall@30**

In [ ]:
def calculate_recall_at_k(retrieved_docs, relevant_docs, k=30):
    true_positives = len(set(retrieved_docs[:k]) & set(relevant_docs))
    recall = true_positives / len(relevant_docs) if relevant_docs else 0
    return recall

*   **F1-score**

In [ ]:
# Fonction pour calculer le F1-score à k
def f1_at_k(precision_at_k, recall_at_k):
    if precision_at_k + recall_at_k == 0:
        return 0
    return 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)

*   **Mean Average Precision (MAP)**

In [ ]:
def mean_average_precision(retrieved, relevant):
    total_queries = len(retrieved)  # Nombre total de requêtes
    sum_avg_precision = 0  # Somme des précisions moyennes pour toutes les requêtes

    for query_results, query_relevants in zip(retrieved, relevant):
        if not query_relevants:  # Si aucun document pertinent, continuer à la prochaine requête
            continue

        relevant_found = 0
        precision_sum = 0
        for rank, doc in enumerate(query_results, start=1):
            if doc in query_relevants:
                relevant_found += 1
                precision_at_rank = relevant_found / rank
                precision_sum += precision_at_rank

        avg_precision = precision_sum / len(query_relevants) if query_relevants else 0
        sum_avg_precision += avg_precision

    mean_avg_precision = sum_avg_precision / total_queries if total_queries > 0 else 0
    return mean_avg_precision

*   **Évaluation de la Recherche**

In [ ]:
# Fonction d'évaluation pour une requête unique
def evaluate_query(model, df, query, n=30, threshold=0.7):

    # Recherche des documents et récupération des documents pertinents
    top_documents, relevant_documents = search_documents(model, df, query, top_n=n, threshold=threshold)

    # Extraire les descriptions des documents récupérés et pertinents
    retrieved_docs = [doc[0] for doc in top_documents]  # Description des documents récupérés
    relevant_docs = [doc[0] for doc in relevant_documents]  # Description des documents pertinents

    # Calcul des métriques d'évaluation
    precision_at_30 = calculate_precision_at_k(retrieved_docs, relevant_docs, k=n)
    recall_at_30 = calculate_recall_at_k(retrieved_docs, relevant_docs, k=n)

    # Calcul du F1 à 30 (en utilisant votre fonction)
    f1_30 = f1_at_k(precision_at_30, recall_at_30)

    # Calcul de la moyenne de la précision (MAP)
    map_score = mean_average_precision([retrieved_docs], [relevant_docs])

    # Affichage des résultats
    print(f"Precision@{n}: {precision_at_30:.4f}")
    print(f"Recall@{n}: {recall_at_30:.4f}")
    print(f"F1-score: {f1_30:.4f}")
    print(f"MAP: {map_score:.4f}")

    return precision_at_30, recall_at_30, f1_30, map_score

In [ ]:
# Exemple de requêtes
query = "Godchild Diana Alacron Flores"

# Calcul des métriques pour la requête
precision_at_30, recall_at_30, f1_30, map_score = evaluate_query(model, data, query, n=3, threshold=0.5)

Precision@3: 1.0000
Recall@3: 0.6000
F1-score: 0.7500
MAP: 0.6000
